<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/Lab_1_Lektion_2_Kontrollera_Roboten.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Städrobot - Lektion 2: Kontrollera roboten

Välkommen till **Lektion 2**! I förra lektionen lärde vi oss om rutnätsvärlden och hur den ser ut.
Nu ska vi lära oss att **styra roboten**!

## 🎯 Vad du lär dig idag:
- Vad är en **simulering**?
- Hur rör sig roboten i rutnätet?
- Vad händer om roboten försöker gå genom en **vägg**?
- Hur **dammsugaren** städar smuts
- Hur **poängsystemet** fungerar
- Styra roboten med **knappar**!

## 📋 Hur du jobbar:
1. Kör varje kodcell uppifrån och ned (tryck **Shift+Enter**)
2. Läs vad som händer noga
3. Fyll i `___` i övningarna
4. Experimentera gärna!

---
*Inga klasser - bara enkla funktioner!*

## Del 1: Förberedelser - Importera verktyg 🛠️

Precis som i Lektion 1 behöver vi importera några verktyg och skapa vår värld.
Kör cellen nedan - den innehåller allt vi behöver!

In [ ]:
# ============================================================
# FÖRBEREDELSER - Kör denna cell FÖRST!
# ============================================================

import ipywidgets as widgets           # Knappar och gränssnitt
from IPython.display import display, clear_output  # Visa/rensa utskrifter

# ---- Celltyper i rutnätet ----
TOMT  = 0   # Tom cell
VÄGG  = 1   # Vägg (kan ej passera)
SMUTS = 2   # Smuts (ska städas)
HEM   = 3   # Robotens startplats

# ---- Symboler för utskrift ----
SYMBOLER = {
    TOMT:  "·",   # Tom cell
    VÄGG:  "█",   # Vägg
    SMUTS: "░",   # Smuts
    HEM:   "H",   # Hem
    "R":   "R",   # Robot
}

# ---- Funktion: Skapa en enkel testvärld ----
def skapa_testvärld():
    # Förklaring: 1=vägg, 0=tomt, 2=smuts, 3=hem
    värld = [
        [1, 1, 1, 1, 1, 1],   # Rad 0 - takväggar
        [1, 3, 0, 2, 0, 1],   # Rad 1 - hem (H) och smuts
        [1, 0, 1, 0, 2, 1],   # Rad 2 - en innervägg
        [1, 2, 0, 0, 0, 1],   # Rad 3 - smuts
        [1, 0, 0, 2, 0, 1],   # Rad 4 - smuts
        [1, 1, 1, 1, 1, 1],   # Rad 5 - golvväggar
    ]
    return värld

# ---- Funktion: Skriva ut rutnätet ----
def visa_värld(värld, robot_x=None, robot_y=None, rubrik=''):
    # Skriver ut rutnätet med symboler. Roboten visas som "R".
    if rubrik:
        print(rubrik)
    print('  ', end='')
    for x in range(len(värld[0])):
        print(f' {x}', end='')
    print()
    for y, rad in enumerate(värld):
        print(f'{y} ', end='')
        for x, cell in enumerate(rad):
            if robot_x == x and robot_y == y:
                print(' R', end='')
            else:
                symbol = SYMBOLER.get(cell, '?')
                print(f' {symbol}', end='')
        print()
    print()

# ---- Funktion: Djupkopiera en värld ----
def kopiera_värld(värld):
    # Skapar en kopia av världen (för att visa FÖRE och EFTER)
    return [rad[:] for rad in värld]

print("✅ Klar! Alla verktyg är laddade.")
print(f"   TOMT={TOMT}, VÄGG={VÄGG}, SMUTS={SMUTS}, HEM={HEM}")
print()
print("Symboler som används:")
namn_map = {0: "Tom cell", 1: "Vägg", 2: "Smuts", 3: "Hem"}
for typ, symbol in SYMBOLER.items():
    if typ != 'R':
        print(f'   {symbol} = {namn_map.get(typ, typ)}')
print("   R = Roboten")


### 👀 Titta på vår testvärld

Kör cellen nedan för att se hur vår testvärld ser ut.
Kom ihåg: `·` = tom, `█` = vägg, `░` = smuts, `H` = hem

In [ ]:
# Skapa testvärlden
värld = skapa_testvärld()

# Visa världen med roboten på startpositionen
robot_x = 1  # Startposition X (kolumn)
robot_y = 1  # Startposition Y (rad)

print("🌍 Vår testvärld (6x6 rutnät):")
print("─" * 20)
visa_värld(värld, robot_x, robot_y, "Startläge:")
print(f"Roboten (R) befinner sig på position: x={robot_x}, y={robot_y}")
print()
print("Smutsceller (░) att städa:")
for y in range(len(värld)):
    for x in range(len(värld[0])):
        if värld[y][x] == SMUTS:
            print(f"   Smuts på position ({x}, {y})")


---
## Del 2: Vad är en simulering? 🎮

**Simulering** = Ett datorprogram som **låtsas** att något händer på riktigt.

I vår simulering:
- Rutnätet är **världen**
- `R` är **roboten**
- Varje gång vi kör kod sker ett **steg** i simuleringen

> **Tänk på det som ett spel:** Varje knapptryckning är ett steg framåt i spelet!

### Vad händer vid varje steg?
1. Roboten gör en **åtgärd** (rörelse eller dammsugning)
2. **Världen uppdateras** (robotens position ändras, smuts försvinner)
3. Vi ser **resultatet** direkt!

Det här kallas **iteration** = ett varv i en loop.

---
## Del 3: Enkel rörelse 🏃

Hur rör sig roboten? Med en enkel funktion!

In [ ]:
# ============================================================
# FUNKTION: move_robot - Flytta roboten
# ============================================================

def move_robot(x, y, riktning):
    # Beräknar ny position för roboten baserat på riktning.
    # x        = Nuvarande X-position (kolumn, ökar åt höger)
    # y        = Nuvarande Y-position (rad, ökar nedåt)
    # riktning = Vilken väg roboten ska gå
    if riktning == "höger":
        ny_x = x + 1  # Öka X med 1 = gå höger
        ny_y = y      # Y ändras inte
    elif riktning == "vänster":
        ny_x = x - 1  # Minska X med 1 = gå vänster
        ny_y = y      # Y ändras inte
    elif riktning == "upp":
        ny_x = x      # X ändras inte
        ny_y = y - 1  # Minska Y med 1 = gå upp (Y ökar nedåt!)
    elif riktning == "ner":
        ny_x = x      # X ändras inte
        ny_y = y + 1  # Öka Y med 1 = gå ner
    else:
        print(f'❓ Okänd riktning: {riktning}')
        ny_x = x  # Inget händer om riktningen är okänd
        ny_y = y
    return ny_x, ny_y

print("✅ move_robot() är redo!")
print()
print("Hur koordinaterna fungerar:")
print("  (0,0) är uppe till vänster")
print("  X ökar åt höger →")
print("  Y ökar nedåt ↓")
print()
print("Exempel:")
print(f"  move_robot(2, 2, 'höger')   → {move_robot(2, 2, 'höger')}")
print(f"  move_robot(2, 2, 'vänster') → {move_robot(2, 2, 'vänster')}")
print(f"  move_robot(2, 2, 'upp')     → {move_robot(2, 2, 'upp')}")
print(f"  move_robot(2, 2, 'ner')     → {move_robot(2, 2, 'ner')}")


### 🔍 Se rörelsen - FÖRE och EFTER

Kör cellen nedan för att se vad som händer när roboten rör sig höger!

In [ ]:
# Demo: Flytta roboten höger - FÖRE och EFTER
värld = skapa_testvärld()
robot_x = 1
robot_y = 1

print("=" * 30)
print("STEG 1: FÖRE rörelsen")
print("=" * 30)
visa_värld(värld, robot_x, robot_y, f"Robot på ({robot_x}, {robot_y}):")

# Flytta roboten höger
ny_x, ny_y = move_robot(robot_x, robot_y, 'höger')

print("=" * 30)
print("STEG 2: EFTER rörelsen")
print("=" * 30)
visa_värld(värld, ny_x, ny_y, f"Robot på ({ny_x}, {ny_y}):")

print("➡️  Rörelse: höger")
print(f"   Position FÖRE: ({robot_x}, {robot_y})")
print(f"   Position EFTER: ({ny_x}, {ny_y})")
print(f"   Förändring: X ändrades från {robot_x} till {ny_x} (+1)")
print(f"   Y ändrades INTE (fortfarande {ny_y})")

robot_x = ny_x
robot_y = ny_y


---
### 🎯 ÖVNING 1: Fyll i riktningarna

Titta på funktionen nedan. Den är **ofullständig** - du måste fylla i `___`!

**Tips:** Titta på hur `move_robot` fungerar ovan. Vad händer med X och Y för varje riktning?

In [ ]:
# ÖVNING 1: Fyll i de saknade värdena
# ------------------------------------
# Ersätt ___ med rätt siffror!

def move_robot_övning(x, y, riktning):
    # Din version av move_robot - fyll i de tomma värdena!
    if riktning == "höger":
        return x + 1, y          # Redan ifylld!
    elif riktning == "vänster":
        return x - ___, y        # ← Fyll i: Vad ska vi ändra X med?
    elif riktning == "upp":
        return x, y - ___        # ← Fyll i: Vad ska vi ändra Y med?
    elif riktning == "ner":
        return x, y + ___        # ← Fyll i: Vad ska vi ändra Y med?
    else:
        return x, y

# Testa din funktion:
print("Testar move_robot_övning:")
print(f"  höger från (2,2):   {move_robot_övning(2, 2, 'höger')}   ← ska bli (3, 2)")
print(f"  vänster från (2,2): {move_robot_övning(2, 2, 'vänster')} ← ska bli (1, 2)")
print(f"  upp från (2,2):     {move_robot_övning(2, 2, 'upp')}     ← ska bli (2, 1)")
print(f"  ner från (2,2):     {move_robot_övning(2, 2, 'ner')}     ← ska bli (2, 3)")
print()
print("✅ Om alla svar stämmer är du klar med Övning 1!")


---
### 🎯 ÖVNING 2: Flytta roboten 5 steg höger

Roboten börjar på position (1, 3). Flytta den **5 steg höger**.
Vilken position är den på nu? Gissa svaret **innan** du kör koden!

In [ ]:
# ÖVNING 2: Flytta 5 steg höger
värld = skapa_testvärld()
robot_x = 1
robot_y = 3

print("Startposition:", (robot_x, robot_y))
visa_värld(värld, robot_x, robot_y, "Start:")

# Flytta 5 steg höger (ett steg i taget)
for steg in range(1, 6):  # steg = 1, 2, 3, 4, 5
    robot_x, robot_y = move_robot(robot_x, robot_y, 'höger')
    print(f'Steg {steg}: Robot på ({robot_x}, {robot_y})')

print()
visa_värld(värld, robot_x, robot_y, "Slutposition:")
print(f"❓ Fråga: Var är roboten nu? ({robot_x}, {robot_y})")
print(f"   Är det som du förväntade dig?")
print()
print("💡 Observera: move_robot kontrollerar INTE väggar!")
print("   Roboten gick utanför rutnätet - det är ett problem!")
print("   I nästa del lär vi oss att fixa detta.")


---
## Del 4: Kollisioner med väggar 🧱

Vad händer om roboten försöker gå in i en **vägg**?

Med `move_robot` räknar vi bara ut den **nya positionen** - vi kollar inte om det är en vägg!

Vi behöver en ny funktion: `can_move()` som kontrollerar om en rörelse är möjlig.

> **Kan du gå genom en vägg på riktigt? Nej!**
> Roboten ska heller inte kunna göra det i simuleringen.

In [ ]:
# ============================================================
# FUNKTION: can_move - Kolla om rörelsen är möjlig
# ============================================================

def can_move(ny_x, ny_y, värld):
    # Kontrollerar om roboten KAN flytta sig till position (ny_x, ny_y).
    # Kontroll 1: Är positionen utanför rutnätets gränser?
    # Kontroll 2: Är positionen en vägg?
    # Returnerar: True (kan röra sig) eller False (kan INTE röra sig)

    antal_kolumner = len(värld[0])  # Antal kolumner (bredd)
    antal_rader    = len(värld)     # Antal rader (höjd)

    # Kontroll 1: Utanför gränserna?
    if ny_x < 0:
        print(f'   🚫 Position ({ny_x}, {ny_y}) är utanför vänstergränsen!')
        return False
    if ny_x >= antal_kolumner:
        print(f'   🚫 Position ({ny_x}, {ny_y}) är utanför högergränsen!')
        return False
    if ny_y < 0:
        print(f'   🚫 Position ({ny_x}, {ny_y}) är utanför övre gränsen!')
        return False
    if ny_y >= antal_rader:
        print(f'   🚫 Position ({ny_x}, {ny_y}) är utanför nedre gränsen!')
        return False

    # Kontroll 2: Är det en vägg?
    if värld[ny_y][ny_x] == VÄGG:
        print(f'   🧱 Position ({ny_x}, {ny_y}) är en vägg!')
        return False

    # Allt OK - roboten kan röra sig dit!
    return True

print("✅ can_move() är redo!")
print()
print("Testar can_move med testvärlden:")
värld = skapa_testvärld()
print(f'  can_move(2, 1, värld) = {can_move(2, 1, värld)}  ← Tom cell, ska vara True')
print(f'  can_move(0, 1, värld) = {can_move(0, 1, värld)}  ← Vägg, ska vara False')
print(f'  can_move(2, 2, värld) = {can_move(2, 2, värld)}  ← Innervägg, ska vara False')


### 🔍 Se kollisionen - Vad händer med väggen?

In [ ]:
# Demo: Försök flytta in i en vägg - FÖRE och EFTER
värld = skapa_testvärld()
robot_x = 1
robot_y = 2

print("=" * 35)
print("Scenario: Robot försöker gå vänster (mot vägg)")
print("=" * 35)
visa_värld(värld, robot_x, robot_y, f"FÖRE - Robot på ({robot_x}, {robot_y}):")

riktning = 'vänster'
ny_x, ny_y = move_robot(robot_x, robot_y, riktning)
print(f'🎯 Försöker gå {riktning} → ny position skulle vara ({ny_x}, {ny_y})')

if can_move(ny_x, ny_y, värld):
    print(f'   ✅ Rörelsen tillåten! Robot flyttar till ({ny_x}, {ny_y})')
    robot_x = ny_x
    robot_y = ny_y
else:
    print(f'   ❌ Rörelsen NEKAD! Robot stannar kvar på ({robot_x}, {robot_y})')

print()
visa_värld(värld, robot_x, robot_y, f"EFTER - Robot på ({robot_x}, {robot_y}):")
print("💡 Observera: Roboten rörde sig INTE! Den är kvar på samma plats.")


---
### 🎯 ÖVNING 3: Fyll i väggkontrollen

Funktionen nedan saknar ett viktigt värde. Fyll i `___`!

**Tips:** Vad är värdet för en vägg? Titta på konstanterna vi definierade: `VÄGG = ?`

In [ ]:
# ÖVNING 3: Fyll i väggkontrollen
# ----------------------------------

def can_move_övning(ny_x, ny_y, värld):
    # Förenklad can_move - fyll i den saknade delen!
    antal_kolumner = len(värld[0])
    antal_rader    = len(värld)

    # Kontroll 1: Utanför gränserna?
    if ny_x < 0 or ny_x >= antal_kolumner:
        return False
    if ny_y < 0 or ny_y >= antal_rader:
        return False

    # Kontroll 2: Är det en vägg?
    # ← Fyll i: Vad är värdet för en vägg? (Titta på VÄGG = ?)
    if värld[ny_y][ny_x] == ___:
        return False

    return True

# Testa din funktion:
värld = skapa_testvärld()
print("Testar can_move_övning:")
print(f'  Tom cell (2,1):     {can_move_övning(2, 1, värld)}   ← ska vara True')
print(f'  Vägg (0,1):         {can_move_övning(0, 1, värld)}  ← ska vara False')
print(f'  Utanför (-1,1):     {can_move_övning(-1, 1, värld)} ← ska vara False')
print()
print("✅ Om alla svar stämmer är du klar med Övning 3!")


In [ ]:
# ============================================================
# FUNKTION: säker_rörelse - Kombinerar move_robot + can_move
# ============================================================

def säker_rörelse(robot_x, robot_y, riktning, värld):
    # Försöker flytta roboten i angiven riktning.
    # Om rörelsen är blockerad (vägg/gräns) stannar roboten kvar.
    # Returnerar: ny_x, ny_y, lyckades

    # Steg 1: Beräkna den önskade nya positionen
    ny_x, ny_y = move_robot(robot_x, robot_y, riktning)

    # Steg 2: Kolla om rörelsen är möjlig
    if can_move(ny_x, ny_y, värld):
        return ny_x, ny_y, True   # Lyckades!
    else:
        return robot_x, robot_y, False  # Misslyckades - stannar kvar

print("✅ säker_rörelse() är redo!")
print()

# Testa
värld = skapa_testvärld()
print("Test 1: Rörelse till tom cell (höger)")
x, y, ok = säker_rörelse(1, 1, 'höger', värld)
print(f'  Resultat: ({x}, {y}), lyckades={ok}')
print()
print("Test 2: Rörelse mot vägg (vänster)")
x, y, ok = säker_rörelse(1, 1, 'vänster', värld)
print(f'  Resultat: ({x}, {y}), lyckades={ok}')


---
### 🎯 ÖVNING 4: Experimentera med väggar

Testa alla fyra riktningar från startpositionen (1, 1).
Vad händer i varje fall? Förutsäg svaret **innan** du kör!

In [ ]:
# ÖVNING 4: Testa alla riktningar från position (1,1)
värld = skapa_testvärld()
start_x = 1
start_y = 1
riktningar = ['höger', 'vänster', 'upp', 'ner']

print(f'Startposition: ({start_x}, {start_y})')
visa_värld(värld, start_x, start_y, "Startläge:")

for riktning in riktningar:
    ny_x, ny_y, lyckades = säker_rörelse(start_x, start_y, riktning, värld)
    if lyckades:
        cell_typ = värld[ny_y][ny_x]
        typ_namn = {TOMT: 'tom', SMUTS: 'smuts', HEM: 'hem'}.get(cell_typ, 'okänd')
        print(f'  {riktning:10} → ({ny_x}, {ny_y}) ✅ (cell är: {typ_namn})')
    else:
        print(f'  {riktning:10} → BLOCKERAD ❌')

print()
print("❓ Frågor att fundera på:")
print("  1. Vilka riktningar var blockerade?")
print("  2. Vad var det som blockerade roboten?")
print("  3. Vad händer om du ändrar startpositionen till (2, 2)?")


---
### 🎯 ÖVNING 5: Förutsäg rörelsen

Titta på rutnätet och svara på frågorna **innan** du kör koden.
Roboten är på position (3, 1). Vad händer om den:
1. Går **höger**?
2. Går **vänster**?
3. Går **upp**?
4. Går **ner**?

Kör sedan koden för att kontrollera!

In [ ]:
# ÖVNING 5: Förutsäg rörelsen
värld = skapa_testvärld()
robot_x = 3
robot_y = 1

visa_värld(värld, robot_x, robot_y, f"Robot på ({robot_x}, {robot_y}):")

print("❓ Förutsäg INNAN du kör:")
print("   1. Vad händer om roboten går höger?   → ___")
print("   2. Vad händer om roboten går vänster? → ___")
print("   3. Vad händer om roboten går upp?     → ___")
print("   4. Vad händer om roboten går ner?     → ___")
print()

# Ändra riktning för att testa olika alternativ
riktning = 'höger'  # ← Ändra denna rad för att testa andra riktningar!
ny_x, ny_y, lyckades = säker_rörelse(robot_x, robot_y, riktning, värld)
print(f'Resultat för {riktning}: ({ny_x}, {ny_y}), lyckades={lyckades}')
visa_värld(värld, ny_x, ny_y, f"EFTER - Robot på ({ny_x}, {ny_y}):")


---
## Del 5: Dammsuga smuts 🧹

Nu ska roboten lära sig att **städa**!

Dammsugaren fungerar så här:
- Om roboten är på en cell med **smuts** → smutsen försvinner ✅
- Om roboten är på en **tom cell** → ingenting händer (men kostar ändå poäng!)

> **Tänk på det som en riktig dammsugare:** Den suger bara upp smuts om det finns smuts!

In [ ]:
# ============================================================
# FUNKTION: dammsug - Dammsug på nuvarande position
# ============================================================

def dammsug(robot_x, robot_y, värld):
    # Om det finns smuts: Smutsen tas bort (cellen blir tom)
    # Om ingen smuts: Ingenting händer
    # Returnerar: städat (True/False), uppdaterad_värld

    # Kolla vad som finns på robotens position
    cell_typ = värld[robot_y][robot_x]

    if cell_typ == SMUTS:
        # Det finns smuts! Städa den.
        värld[robot_y][robot_x] = TOMT  # Ersätt smuts med tom cell
        print(f'   🧹 Städade smuts på ({robot_x}, {robot_y})!')
        return True, värld
    else:
        # Ingen smuts här
        print(f'   💨 Ingen smuts på ({robot_x}, {robot_y}) - inget att städa')
        return False, värld

print("✅ dammsug() är redo!")
print()
print("Testar dammsug:")
testvärld = skapa_testvärld()
print(f'  Cell (3,1) är: {testvärld[1][3]} (SMUTS={SMUTS})')
städat, testvärld = dammsug(3, 1, testvärld)
print(f'  Cell (3,1) är nu: {testvärld[1][3]} (TOMT={TOMT}) - smutsen städades!')


### 🔍 Se dammsugningen - FÖRE och EFTER

In [ ]:
# Demo: Dammsugning - FÖRE och EFTER
värld = skapa_testvärld()
robot_x = 3
robot_y = 1

print("=" * 35)
print("Scenario 1: Dammsug på SMUTS-cell")
print("=" * 35)
visa_värld(värld, robot_x, robot_y, f"FÖRE - Robot på ({robot_x}, {robot_y}):")
print(f'Celltyp på ({robot_x},{robot_y}): {värld[robot_y][robot_x]} = SMUTS')
print()

städat, värld = dammsug(robot_x, robot_y, värld)

print()
visa_värld(värld, robot_x, robot_y, f"EFTER - Robot på ({robot_x}, {robot_y}):")
print(f'Celltyp på ({robot_x},{robot_y}): {värld[robot_y][robot_x]} = TOMT')
print()
print("💡 Observera: Smutsen (░) försvann! Cellen är nu tom (·)")

print()
print("=" * 35)
print("Scenario 2: Dammsug på TOM cell")
print("=" * 35)
visa_värld(värld, robot_x, robot_y, f"FÖRE - Robot på ({robot_x}, {robot_y}):")

städat, värld = dammsug(robot_x, robot_y, värld)

print()
visa_värld(värld, robot_x, robot_y, f"EFTER - Robot på ({robot_x}, {robot_y}):")
print()
print("💡 Observera: Ingenting ändrades! Cellen var redan tom.")


---
### 🎯 ÖVNING 6: Hitta och städa all smuts

Skapa kod som hittar alla smutsceller och städar dem!

In [ ]:
# ÖVNING 6: Hitta och städa all smuts
värld = skapa_testvärld()

# Hitta alla smutsceller
smuts_lista = []
for y in range(len(värld)):
    for x in range(len(värld[0])):
        if värld[y][x] == SMUTS:
            smuts_lista.append((x, y))

print(f'Hittade {len(smuts_lista)} smutsceller:')
for (sx, sy) in smuts_lista:
    print(f'   ░ Smuts på ({sx}, {sy})')

visa_värld(värld, 1, 1, "\nFÖRE städning:")

# Städa ALL smuts (roboten teleporterar sig magiskt)
for (sx, sy) in smuts_lista:
    städat, värld = dammsug(sx, sy, värld)

print()
visa_värld(värld, 1, 1, "EFTER städning:")

kvarv = sum(1 for y in range(len(värld)) for x in range(len(värld[0])) if värld[y][x] == SMUTS)
print(f'Kvarvarande smuts: {kvarv} celler')
print("✅ All smuts är städad!" if kvarv == 0 else "⚠️  Det finns fortfarande smuts kvar!")


---
### 🎯 ÖVNING 7: Fyll i dammsugningsfunktionen

Funktionen nedan är ofullständig. Fyll i `___`!

In [ ]:
# ÖVNING 7: Fyll i dammsugningsfunktionen

def dammsug_övning(robot_x, robot_y, värld):
    # Din version av dammsug - fyll i de tomma värdena!
    cell_typ = värld[robot_y][robot_x]

    if cell_typ == ___:                # ← Fyll i: Vilket värde är smuts? (SMUTS = ?)
        värld[robot_y][robot_x] = ___  # ← Fyll i: Vad ska cellen bli efter städning? (TOMT = ?)
        return True, värld
    else:
        return False, värld

# Testa din funktion:
testvärld = skapa_testvärld()
print(f'Testar med smutscell (3,1): {testvärld[1][3]} (ska vara {SMUTS}=SMUTS)')
städat, testvärld = dammsug_övning(3, 1, testvärld)
print(f'Städades: {städat}  ← ska vara True')
print(f'Cell (3,1) är nu: {testvärld[1][3]}  ← ska vara {TOMT} (TOMT)')
print()
print("✅ Om alla svar stämmer är du klar med Övning 7!")


---
## Del 6: Poängsystem 🏆

Roboten samlar poäng baserat på vad den gör:

| Åtgärd | Poäng |
|--------|-------|
| Rörelse (höger/vänster/upp/ner) | **-1 poäng** |
| Dammsugning med smuts | **+100 poäng** |
| Dammsugning utan smuts | **-1 poäng** |

> **Varför minus för rörelse?** Roboten använder ström för att röra sig!
> Vi vill att roboten städar **effektivt** - inte slösar energi!

**Startpoäng: 0**

In [ ]:
# ============================================================
# FUNKTION: uppdatera_poäng - Håll koll på poängen
# ============================================================

def uppdatera_poäng(åtgärd, hittade_smuts, nuvarande_poäng):
    # Beräknar ny poäng baserat på vilken åtgärd som gjordes.
    # åtgärd         = 'rör' (rörelse) eller 'dammsug'
    # hittade_smuts  = True om dammsugning hittade smuts
    # nuvarande_poäng = Poängen innan åtgärden
    # Returnerar: ny_poäng, förändring

    if åtgärd == 'rör':
        förändring = -1      # Rörelse kostar 1 poäng
    elif åtgärd == 'dammsug':
        if hittade_smuts:
            förändring = +100  # Städad smuts: +100 poäng!
        else:
            förändring = -1    # Tom cell: -1 poäng (onödig dammsugning)
    else:
        förändring = 0  # Okänd åtgärd

    ny_poäng = nuvarande_poäng + förändring
    return ny_poäng, förändring

print("✅ uppdatera_poäng() är redo!")
print()
print("Testar poängsystemet:")
poäng = 0
print(f'  Startpoäng: {poäng}')
poäng, d = uppdatera_poäng('rör', False, poäng)
print(f'  Efter rörelse:               poäng={poäng} (förändring: {d:+d})')
poäng, d = uppdatera_poäng('dammsug', True, poäng)
print(f'  Efter dammsugning (smuts):   poäng={poäng} (förändring: {d:+d})')
poäng, d = uppdatera_poäng('dammsug', False, poäng)
print(f'  Efter dammsugning (tomt):    poäng={poäng} (förändring: {d:+d})')
for _ in range(3):
    poäng, d = uppdatera_poäng('rör', False, poäng)
print(f'  Efter 3 fler rörelser:       poäng={poäng}')


---
### 🎯 ÖVNING 8: Fyll i poängvärdena

Fyll i de rätta poängvärdena i funktionen!

In [ ]:
# ÖVNING 8: Fyll i poängberäkningen

def uppdatera_poäng_övning(åtgärd, hittade_smuts, nuvarande_poäng):
    # Din version - fyll i poängvärdena!
    if åtgärd == 'rör':
        förändring = ___     # ← Fyll i: Hur mycket kostar en rörelse?
    elif åtgärd == 'dammsug':
        if hittade_smuts:
            förändring = ___ # ← Fyll i: Bonus för att städa smuts?
        else:
            förändring = ___ # ← Fyll i: Kostnad för onödig dammsugning?
    else:
        förändring = 0

    return nuvarande_poäng + förändring, förändring

# Testa din funktion:
poäng = 0
poäng, d = uppdatera_poäng_övning('rör', False, poäng)
print(f'Efter rörelse:             poäng={poäng}  ← ska vara -1')
poäng, d = uppdatera_poäng_övning('dammsug', True, poäng)
print(f'Efter dammsug (smuts):     poäng={poäng} ← ska vara 99')
poäng, d = uppdatera_poäng_övning('dammsug', False, poäng)
print(f'Efter dammsug (tomt):      poäng={poäng} ← ska vara 98')
print()
print("✅ Om alla svar stämmer är du klar med Övning 8!")


---
### 🎯 ÖVNING 9: Beräkna poängen manuellt

Roboten gör följande åtgärder (i ordning):
1. Rör höger
2. Rör höger
3. Dammsug (det finns smuts här)
4. Rör ner
5. Dammsug (ingen smuts här)

Beräkna slutpoängen **utan** att köra koden. Skriv ditt svar här: `___`

Kör sedan koden för att kontrollera!

In [ ]:
# ÖVNING 9: Beräkna poängen steg för steg
åtgärder = [
    ('rör',     False),  # Rörelse höger
    ('rör',     False),  # Rörelse höger
    ('dammsug', True),   # Dammsug - smuts!
    ('rör',     False),  # Rörelse ner
    ('dammsug', False),  # Dammsug - tomt
]

print("Beräkning steg för steg:")
poäng = 0
for i, (åtgärd, smuts) in enumerate(åtgärder, 1):
    poäng, förändring = uppdatera_poäng(åtgärd, smuts, poäng)
    smuts_info = '(smuts)' if smuts else '(tomt)' if åtgärd == 'dammsug' else '      '
    print(f'  Åtgärd {i}: {åtgärd:8} {smuts_info} → förändring: {förändring:+d}, poäng: {poäng}')

print()
print(f'✅ Slutpoäng: {poäng}')
print("   Stämde det med ditt gissade svar?")


---
## Del 7: Komplett simulering 🎮

Nu sätter vi ihop allt: rörelse + väggkontroll + dammsugning + poäng!

In [ ]:
# ============================================================
# KOMPLETT SIMULERING - Kombinerar allt!
# ============================================================

def kör_simulering(åtgärder, visa_varje_steg=True):
    # Kör en sekvens av åtgärder med roboten.
    # åtgärder = Lista med ('typ', parameter) tupler
    # Returnerar slutpoäng

    värld   = skapa_testvärld()
    robot_x = 1
    robot_y = 1
    poäng   = 0

    print(f'🚀 Simulering startar! Position: ({robot_x}, {robot_y}), Poäng: {poäng}')
    if visa_varje_steg:
        visa_värld(värld, robot_x, robot_y, 'START:')

    for steg_nr, (åtgärd, parameter) in enumerate(åtgärder, 1):
        print(f'\nSteg {steg_nr}: {åtgärd} {parameter or ""}')

        if åtgärd == 'rör':
            ny_x, ny_y, lyckades = säker_rörelse(robot_x, robot_y, parameter, värld)
            if lyckades:
                robot_x, robot_y = ny_x, ny_y
                poäng, _ = uppdatera_poäng('rör', False, poäng)
                print(f'   ✅ Rörde sig {parameter} → ({robot_x}, {robot_y}), Poäng: {poäng}')
            else:
                print(f'   ❌ Kan ej röra {parameter} - stannar på ({robot_x}, {robot_y})')

        elif åtgärd == 'dammsug':
            städat, värld = dammsug(robot_x, robot_y, värld)
            poäng, förändring = uppdatera_poäng('dammsug', städat, poäng)
            emoji = '✅' if städat else '💨'
            resultat = 'städade smuts!' if städat else 'ingenting (tomt)'
            print(f'   {emoji} {resultat}, poäng {förändring:+d} → {poäng}')

        if visa_varje_steg:
            visa_värld(värld, robot_x, robot_y, f'  Efter steg {steg_nr}:')

    print()
    print('=' * 35)
    print(f'🏁 Simulering klar! Slutpoäng: {poäng}')
    kvarv = sum(1 for y in range(len(värld)) for x in range(len(värld[0])) if värld[y][x] == SMUTS)
    print(f'   Kvarvarande smuts: {kvarv} celler')
    print('=' * 35)
    return poäng

# Exempelkörning
print("EXEMPELKÖRNING:")
print("─" * 35)
exempel = [
    ('rör',     'höger'),
    ('rör',     'höger'),
    ('dammsug', None),
    ('rör',     'ner'),
    ('rör',     'höger'),
    ('dammsug', None),
]
kör_simulering(exempel, visa_varje_steg=False)


---
### 🎯 ÖVNING 10: Planera din städrutt!

Titta på rutnätet och planera en rutt som städar **alla** smutsceller.
Fyll i din åtgärdslista och kör simuleringen!

In [ ]:
# ÖVNING 10: Planera din städrutt!
värld_kopia = skapa_testvärld()
visa_värld(värld_kopia, 1, 1, "Startläge (R=robot på (1,1)):")
print("Smutsceller att städa:")
for y in range(len(värld_kopia)):
    for x in range(len(värld_kopia[0])):
        if värld_kopia[y][x] == SMUTS:
            print(f'   ░ Smuts på ({x}, {y})')
print()
print("Din uppgift: Fyll i åtgärdslistan nedan för att städa ALLA smutsceller!")
print("Format: ('rör', 'riktning') eller ('dammsug', None)")
print()

# Fyll i din åtgärdslista här!
min_lista = [
    ('rör',     'höger'),   # Steg 1: Flytta höger
    ('rör',     'höger'),   # Steg 2: Flytta höger
    ('dammsug', None),      # Steg 3: Dammsug
    # Lägg till fler steg här!
    # ('rör', '___'),
    # ('dammsug', None),
]

print("Kör din simulering:")
print("─" * 35)
kör_simulering(min_lista, visa_varje_steg=False)


---
### 🎯 ÖVNING 11: Minimera antalet åtgärder

Kan du städa **alla 4 smutsceller** med **färre än 15 åtgärder**?

Smutscellerna finns på: **(3,1), (4,2), (1,3), (3,4)**
Roboten startar på: **(1,1)**

Planera en optimal rutt och mät hur många åtgärder du använde!

> **Tips om optimal rutt:** Smutscellerna är på (3,1), (4,2), (1,3), (3,4). En möjlig minimal rutt är: höger×2 → dammsug → ner×1 → höger×1 → dammsug → vänster×2 → ner×2 → dammsug → höger×2 → ner×1 → dammsug = ~13 åtgärder. Kan du slå det?


In [ ]:
# ÖVNING 11: Optimal städning
# ----------------------------

optimal_lista = [
    # Fyll i din bästa rutt här!
    # ('rör', '___'),
    # ('dammsug', None),
]

if not optimal_lista:
    print('⚠️  Fyll i optimal_lista med dina åtgärder!')
    print('   Smutscellerna finns på: (3,1), (4,2), (1,3), (3,4)')
    print('   Roboten startar på: (1,1)')
    värld_kopia = skapa_testvärld()
    visa_värld(värld_kopia, 1, 1, 'Karta:')
else:
    print(f'Du har planerat {len(optimal_lista)} åtgärder.')
    slutpoäng = kör_simulering(optimal_lista, visa_varje_steg=False)
    print()
    if len(optimal_lista) < 15:
        print('🌟 Bra jobbat! Under 15 åtgärder!')
    elif len(optimal_lista) < 20:
        print('👍 Bra! Kan du komma under 15?')
    else:
        print('💡 Kan du hitta en kortare rutt?')


---
### 🎯 ÖVNING 12: Förstå gränskontrollen

`can_move()` kontrollerar om roboten är utanför rutnätets gränser.
Fyll i de saknade värdena!

In [ ]:
# ÖVNING 12: Gränskontroll

def can_move_fullständig(ny_x, ny_y, värld):
    # Fullständig gräns- och väggkontroll - fyll i de saknade värdena!
    antal_kolumner = len(värld[0])
    antal_rader    = len(värld)

    # Kontroll 1: Utanför vänstergränsen?
    if ny_x < ___:           # ← Fyll i: Lägsta tillåtna X-värdet
        return False

    # Kontroll 2: Utanför högergränsen?
    if ny_x >= ___:          # ← Fyll i: Vad är max antal kolumner?
        return False

    # Kontroll 3: Utanför övre gränsen?
    if ny_y < ___:           # ← Fyll i: Lägsta tillåtna Y-värdet
        return False

    # Kontroll 4: Utanför nedre gränsen?
    if ny_y >= ___:          # ← Fyll i: Vad är max antal rader?
        return False

    # Kontroll 5: Är det en vägg?
    if värld[ny_y][ny_x] == VÄGG:
        return False

    return True

# Testa:
värld = skapa_testvärld()
print("Testar can_move_fullständig:")
print(f'  (2,2):   {can_move_fullständig(2, 2, värld)}  ← ska vara False (innervägg)')
print(f'  (2,3):   {can_move_fullständig(2, 3, värld)}  ← ska vara True (tom cell)')
print(f'  (-1,1):  {can_move_fullständig(-1, 1, värld)} ← ska vara False (utanför)')
print(f'  (6,1):   {can_move_fullständig(6, 1, värld)}  ← ska vara False (utanför)')
print()
print("✅ Om alla svar stämmer är du klar med Övning 12!")


---
### 🎯 ÖVNING 13: Poängkalkyl

Roboten ska städa 3 smutsceller.
Räkna ut: Vad är den **bästa möjliga slutpoängen** om den behöver 8 rörelser?

In [ ]:
# ÖVNING 13: Beräkna maximal poäng

antal_rörelser   = 8
antal_städningar = 3
antal_tomma_sug  = 0

# Fyll i poängvärdena:
poäng_per_rörelse   = ___  # ← Fyll i: Hur mycket kostar en rörelse?
poäng_per_städning  = ___  # ← Fyll i: Bonus för lyckad städning?
poäng_per_tomt_sug  = ___  # ← Fyll i: Kostnad för onödig dammsugning?

total = (antal_rörelser   * poäng_per_rörelse  +
         antal_städningar * poäng_per_städning +
         antal_tomma_sug  * poäng_per_tomt_sug)

print(f'Rörelser:         {antal_rörelser} × {poäng_per_rörelse} = {antal_rörelser * poäng_per_rörelse}')
print(f'Lyckad städning:  {antal_städningar} × {poäng_per_städning} = {antal_städningar * poäng_per_städning}')
print(f'Onödig städning:  {antal_tomma_sug} × {poäng_per_tomt_sug} = {antal_tomma_sug * poäng_per_tomt_sug}')
print(f'─' * 30)
print(f'Total poäng:      {total}')
print()
print("❓ Stämmer det med vad du förväntade dig?")


---
### 🎯 ÖVNING 14: Skriv ett komplett spelloop

Fyll i den saknade koden!

Roboten ska:
1. Röra sig 2 steg **höger**
2. Dammsuga
3. Röra sig 1 steg **ner**
4. Dammsuga

In [ ]:
# ÖVNING 14: Komplett spelloop
värld   = skapa_testvärld()
robot_x = 1
robot_y = 1
poäng   = 0

visa_värld(värld, robot_x, robot_y, "START:")

# Steg 1 & 2: Rör 2 steg höger
for _ in range(2):
    ny_x, ny_y, ok = säker_rörelse(robot_x, robot_y, '___', värld)  # ← Fyll i riktning
    if ok:
        robot_x, robot_y = ny_x, ny_y
        poäng, _ = uppdatera_poäng('rör', False, poäng)

print(f'Efter 2 rörelser höger: ({robot_x}, {robot_y}), poäng: {poäng}')
visa_värld(värld, robot_x, robot_y)

# Steg 3: Dammsug
städat, värld = ___  # ← Fyll i: Anropa dammsug() med rätt parametrar
poäng, _ = uppdatera_poäng('dammsug', städat, poäng)
print(f'Efter dammsugning: städat={städat}, poäng: {poäng}')
visa_värld(värld, robot_x, robot_y)

# Steg 4: Rör 1 steg ner
ny_x, ny_y, ok = säker_rörelse(robot_x, robot_y, '___', värld)  # ← Fyll i riktning
if ok:
    robot_x, robot_y = ny_x, ny_y
    poäng, _ = uppdatera_poäng('rör', False, poäng)

# Steg 5: Dammsug
städat, värld = dammsug(robot_x, robot_y, värld)
poäng, _ = uppdatera_poäng('dammsug', städat, poäng)
print(f'Slutpoäng: {poäng}')
visa_värld(värld, robot_x, robot_y, "SLUTLÄGE:")


---
### 🎯 ÖVNING 15: Bygg din egen städfunktion

Skriv klart `gå_och_städa()` som rör roboten mot ett mål och dammsuger längs vägen.

**Tips för att välja riktning mot målet:**
- Om `mål_x > robot_x` → gå **höger**
- Om `mål_x < robot_x` → gå **vänster**
- Om `mål_y > robot_y` → gå **ner**
- Om `mål_y < robot_y` → gå **upp**

In [ ]:
# ÖVNING 15: Bygg en städfunktion

def gå_och_städa(robot_x, robot_y, mål_x, mål_y, värld, poäng):
    # Rör roboten mot målet och dammsuger längs vägen.
    max_steg = 20
    steg = 0

    while (robot_x != mål_x or robot_y != mål_y) and steg < max_steg:
        steg += 1

        # Välj riktning mot målet
        if mål_x > robot_x:
            riktning = 'höger'
        elif mål_x < robot_x:
            riktning = '___'    # ← Fyll i
        elif mål_y > robot_y:
            riktning = '___'    # ← Fyll i
        else:
            riktning = 'upp'

        # Gör rörelsen
        ny_x, ny_y, ok = säker_rörelse(robot_x, robot_y, riktning, värld)
        if ok:
            robot_x, robot_y = ny_x, ny_y
            poäng, _ = uppdatera_poäng('rör', False, poäng)

            # Dammsug om det finns smuts
            if värld[robot_y][robot_x] == SMUTS:
                städat, värld = dammsug(robot_x, robot_y, värld)
                poäng, _ = uppdatera_poäng('dammsug', städat, poäng)
        else:
            break  # Fastnade i vägg

    return robot_x, robot_y, värld, poäng

# Testa din funktion
print("Testar gå_och_städa:")
värld = skapa_testvärld()
robot_x, robot_y, poäng = 1, 1, 0

visa_värld(värld, robot_x, robot_y, "Start:")
print(f'Startposition: ({robot_x}, {robot_y}), Poäng: {poäng}')

# Gå till smuts på (3,1)
robot_x, robot_y, värld, poäng = gå_och_städa(robot_x, robot_y, 3, 1, värld, poäng)
print(f'Vid mål (3,1): Position=({robot_x},{robot_y}), Poäng={poäng}')
visa_värld(värld, robot_x, robot_y)


---
## Del 8: Interaktiv knappkontroll 🎮

Nu sätter vi ihop **allt** i ett interaktivt gränssnitt!

Styr roboten med knappar och se vad som händer i realtid.

**Knappar:**
- ⬆️ ⬇️ ⬅️ ➡️ = Rör roboten
- 🧹 = Dammsug
- 🔄 = Återställ spelet

In [ ]:
# ============================================================
# DEL 8: INTERAKTIV KNAPPKONTROLL
# ============================================================

# Spelstatus som alla knappar kan läsa och ändra
spel = {
    "värld":      skapa_testvärld(),
    "robot_x":    1,
    "robot_y":    1,
    "poäng":      0,
    "åtgärder":   0,
    "meddelande": "Välkommen! Använd knapparna för att styra roboten."
}

def räkna_smuts(värld):
    return sum(1 for y in range(len(värld)) for x in range(len(värld[0])) if värld[y][x] == SMUTS)

def visa_spel(utdata_widget):
    with utdata_widget:
        clear_output(wait=True)
        print('=' * 40)
        print('🤖 STÄDROBOT - MANUELL STYRNING')
        print('=' * 40)
        v  = spel['värld']
        rx = spel['robot_x']
        ry = spel['robot_y']
        print('  ', end='')
        for x in range(len(v[0])):
            print(f' {x}', end='')
        print()
        for y, rad in enumerate(v):
            print(f'{y} ', end='')
            for x, cell in enumerate(rad):
                if rx == x and ry == y:
                    print(' R', end='')
                else:
                    print(f' {SYMBOLER.get(cell, "?")}', end='')
            print()
        print()
        print(f'📍 Position:   ({rx}, {ry})')
        print(f'🏆 Poäng:      {spel["poäng"]}')
        print(f'📊 Åtgärder:   {spel["åtgärder"]}')
        print(f'🧹 Kvar smuts: {räkna_smuts(spel["värld"])} celler')
        print()
        print(f'💬 {spel["meddelande"]}')
        if räkna_smuts(spel['värld']) == 0:
            print()
            print('🎉 GRATTIS! Du städade all smuts!')
            print(f'   Slutpoäng: {spel["poäng"]} på {spel["åtgärder"]} åtgärder')

def knapp_rörelse(riktning, utdata):
    ny_x, ny_y, ok = säker_rörelse(spel['robot_x'], spel['robot_y'], riktning, spel['värld'])
    if ok:
        spel['robot_x'] = ny_x
        spel['robot_y'] = ny_y
        spel['poäng'], _ = uppdatera_poäng('rör', False, spel['poäng'])
        spel['åtgärder'] += 1
        spel['meddelande'] = f'Rörde sig {riktning} → ({ny_x}, {ny_y}). Poäng: {spel["poäng"]}'
    else:
        spel['meddelande'] = f'❌ Kan ej gå {riktning} - blockerad!'
    visa_spel(utdata)

def knapp_dammsug(utdata):
    städat, spel['värld'] = dammsug(spel['robot_x'], spel['robot_y'], spel['värld'])
    spel['poäng'], förändring = uppdatera_poäng('dammsug', städat, spel['poäng'])
    spel['åtgärder'] += 1
    if städat:
        spel['meddelande'] = f'🧹 Städade smuts! Poäng: {förändring:+d} → {spel["poäng"]}'
    else:
        spel['meddelande'] = f'💨 Ingen smuts här. Poäng: {förändring:+d} → {spel["poäng"]}'
    visa_spel(utdata)

def knapp_återställ(utdata):
    spel['värld']      = skapa_testvärld()
    spel['robot_x']    = 1
    spel['robot_y']    = 1
    spel['poäng']      = 0
    spel['åtgärder']   = 0
    spel['meddelande'] = 'Spelet återställt! Börja om.'
    visa_spel(utdata)

# Knappar
btn_layout = widgets.Layout(width='60px', height='45px')
k_upp    = widgets.Button(description='⬆️',  layout=btn_layout)
k_ner    = widgets.Button(description='⬇️',  layout=btn_layout)
k_vänst  = widgets.Button(description='⬅️',  layout=btn_layout)
k_höger  = widgets.Button(description='➡️',  layout=btn_layout)
k_sug    = widgets.Button(description='🧹',  layout=btn_layout, button_style='success')
k_reset  = widgets.Button(description='🔄 Återställ',
                          layout=widgets.Layout(width='120px', height='45px'),
                          button_style='warning')
utdata = widgets.Output()

k_upp.on_click(  lambda b: knapp_rörelse('upp',      utdata))
k_ner.on_click(  lambda b: knapp_rörelse('ner',      utdata))
k_vänst.on_click(lambda b: knapp_rörelse('vänster',  utdata))
k_höger.on_click(lambda b: knapp_rörelse('höger',    utdata))
k_sug.on_click(  lambda b: knapp_dammsug(utdata))
k_reset.on_click(lambda b: knapp_återställ(utdata))

# Layout: pil-knappar i kors-form
tom = widgets.Label('  ')
pil_layout = widgets.VBox([
    widgets.HBox([tom,      k_upp,   tom]),
    widgets.HBox([k_vänst,  k_sug,   k_höger]),
    widgets.HBox([tom,      k_ner,   tom]),
    widgets.HBox([k_reset]),
])

print("🎮 Interaktiv styrning laddad!")
print("   ⬆️ ⬇️ ⬅️ ➡️ = Rör roboten     🧹 = Dammsug     🔄 = Återställ")
print()
display(widgets.HBox([pil_layout, utdata]))
visa_spel(utdata)


---
### 🎯 UPPDRAG: Städa med minimala åtgärder!

Använd knapparna ovan för att:

1. **Grunduppdrag:** Städa **all smuts** i rutnätet
2. **Utmaningsnivå 1:** Städa all smuts på **färre än 20 åtgärder**
3. **Utmaningsnivå 2:** Städa all smuts på **färre än 15 åtgärder**
4. **Expertuppdrag:** Maximera poängen - städa allt med **minst rörelser möjligt**!

> **Tips:** Planera rutten INNAN du börjar klicka!
> Kolla var all smuts finns och tänk ut den kortaste vägen.

---
## 📝 Quiz: Testa dina kunskaper!

Svara på frågorna nedan (utan att titta i koden!):

### Fråga 1: Koordinatsystem
Rutnätet har koordinater (x, y). Om roboten är på (3, 2) och rör sig **höger**:
- a) (4, 2)
- b) (3, 3)
- c) (2, 2)
- d) (3, 1)

### Fråga 2: Väggar
Vad händer om roboten försöker gå in i en vägg?
- a) Roboten förstörs
- b) Roboten hoppar över väggen
- c) Roboten stannar kvar - rörelsen nekas
- d) Väggen försvinner

### Fråga 3: Dammsugning
Vad händer om roboten dammsuger på en **tom cell** (ingen smuts)?
- a) Roboten städar ändå - tar bort tomheten
- b) Ingenting händer med cellen - roboten förlorar 1 poäng
- c) Roboten får +100 poäng
- d) Simuleringen kraschar

### Fråga 4: Poängsystem
Roboten rör sig 5 gånger och dammsuger 2 smutsceller. Vad är slutpoängen?
- a) -3 poäng
- b) 195 poäng
- c) 100 poäng
- d) -7 poäng

### Fråga 5: Simulering
Vad är en "simulering"?
- a) En riktig robot som städar ett riktigt rum
- b) En datormodell som låtsas att något händer
- c) En typ av algoritm
- d) En klass i Python

In [ ]:
# Quiz-svar - kör för att kontrollera!
quiz_svar = {
    1: ('a', 'X ökar med 1 vid höger → (3+1, 2) = (4, 2)'),
    2: ('c', 'can_move() returnerar False → roboten stannar kvar'),
    3: ('b', 'Cellen ändras inte, men dammsug kostar -1 poäng'),
    4: ('b', f'5×(-1) + 2×(+100) = {5*(-1) + 2*100} poäng'),
    5: ('b', 'Simulering = datormodell av verkligheten'),
}
print("📝 QUIZ-SVAR:")
print("─" * 45)
for fråga, (svar, förklaring) in quiz_svar.items():
    print(f'Fråga {fråga}: {svar.upper()} - {förklaring}')
print()
print("Hur många fick du rätt? 🌟")


---
## 🎉 Sammanfattning - Lektion 2 klar!

Du har lärt dig att:

✅ **Flytta roboten** med `move_robot(x, y, riktning)`  
✅ **Kontrollera väggar** med `can_move(ny_x, ny_y, värld)`  
✅ **Säker rörelse** med `säker_rörelse()` som kombinerar båda  
✅ **Dammsuga smuts** med `dammsug(robot_x, robot_y, värld)`  
✅ **Hålla koll på poäng** med `uppdatera_poäng(åtgärd, smuts, poäng)`  
✅ **Interaktiv kontroll** med knappar

### 🔢 Poängsystemet:
| Åtgärd | Poäng |
|--------|-------|
| Rörelse | -1 |
| Dammsug (smuts) | +100 |
| Dammsug (tomt) | -1 |

---

### 🚀 Nästa lektion: Robotens sinnen

I Lektion 3 lär vi oss:
- Vad **roboten kan "se"** (percept)
- Hur roboten fattar **enkla beslut** automatiskt
- Introducera **regelbaserad AI**: "Om jag ser smuts → städa"

---
*Städrobot-laborationen - Lektion 2 av 5*  
*Inga klasser används - bara enkla funktioner!*